# 08 — Modelo final para la app (Persona B)

**Problema:** `elo_diff = home_elo + 60 − away_elo`. Con las tres columnas, la logística con todas las features
(`logistica_todas`, notebook 06) repite información y el efecto del Elo se reparte entre tres coeficientes,
así que los factores no se pueden explicar bien.

**Prueba:** misma logística (`tune_logistic`, C elegido con `TimeSeriesSplit` dentro de train) con las
`FEATURE_COLS` **sin `home_elo` ni `away_elo`** → `logistica_sin_redundancia`.

**Regla de decisión (fijada antes de mirar el resultado):** si su log loss en validación queda a menos de
0.005 de `logistica_todas`, es el modelo final; si no, se queda `logistica_todas`.

## 1. Datos y corte temporal (mismos partidos que las cuotas)

In [ ]:
import sys; sys.path.append("..")
import pandas as pd

from src.splits import split
from src.features.build import FEATURE_COLS
from src.models.logistic import tune_logistic, coef_table
from src.models.final import FEATURES_FINAL, cargar_modelo
from src.features.current import matchup_features
from src.evaluation.metrics import evaluate, to_hda
from src.evaluation.registro import guardar_resultado

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])
train, val = split(df)
val_ok = val[val[["odds_H", "odds_D", "odds_A"]].notna().all(axis=1)]
y = val_ok["result"]
cols_sin = [c for c in FEATURE_COLS if c not in ("home_elo", "away_elo")]
assert cols_sin == FEATURES_FINAL
print(f"train: {len(train)} | val con cuotas: {len(val_ok)}")
print("Features sin redundancia:", cols_sin)

## 2. Ajuste en train y evaluación en validación

In [ ]:
modelos = {"logistica_todas": FEATURE_COLS, "logistica_sin_redundancia": cols_sin}
busquedas, res = {}, {}
for nombre, cols in modelos.items():
    busquedas[nombre] = s = tune_logistic(train, cols)
    m = s.best_estimator_
    res[nombre] = evaluate(y, to_hda(m, m.predict_proba(val_ok[cols])), nombre)
    print(f"{nombre}: C = {s.best_params_['logistica__C']:.4g} | log loss CV en train = {-s.best_score_:.4f}")

tabla = guardar_resultado(res["logistica_sin_redundancia"])
tabla.sort_values("log_loss").reset_index(drop=True)

## 3. Decisión

In [ ]:
dif = res["logistica_sin_redundancia"]["log_loss"] - res["logistica_todas"]["log_loss"]
final = "logistica_sin_redundancia" if dif < 0.005 else "logistica_todas"
print(f"log loss sin redundancia − todas = {dif:+.4f}  →  modelo final: {final}")

## 4. Coeficientes del modelo sin redundancia
Ahora el Elo entra en un solo coeficiente. Los de forma siguen siendo difíciles de leer uno por uno:
puntos, goles a favor y goles en contra de los últimos 5 partidos están correlacionados entre sí y con el
Elo, así que algunos signos salen contraintuitivos (por ejemplo, más puntos recientes del visitante suben un
poco H). Cada efecto es *manteniendo fijas las demás variables*, y son efectos pequeños frente al del Elo.

In [ ]:
t = coef_table(busquedas["logistica_sin_redundancia"].best_estimator_, cols_sin)
t.assign(**{"H − A": t["H"] - t["A"]}).sort_values("H − A", key=abs, ascending=False).round(3)

## 5. El modelo final en la app
`src/models/final.py` reentrena esta configuración con **train + validación** (2017-18 a 2023-24, sin el test)
y la guarda en `models/modelo_final.joblib`. La app la usa con el contrato `predict_proba(X)` → (n, 3) en
orden H, D, A, y con `factores()` para explicar un partido.

In [ ]:
modelo = cargar_modelo()
print({k: v for k, v in modelo.meta.items() if k != "features"})
for local, visita in [("Liverpool", "Southampton"), ("Arsenal", "Manchester City")]:
    X = matchup_features(df, local, visita)
    p = modelo.predict_proba(X)[0]
    print(f"\n{local} vs {visita}: H={p[0]:.3f}  D={p[1]:.3f}  A={p[2]:.3f}")
    display(pd.DataFrame(modelo.factores(X)).round(3))

## 6. Conclusión

- **Decisión: el modelo final es `logistica_sin_redundancia`.** Su log loss en validación es 0.9258, frente a
  0.9277 de `logistica_todas` (−0.002). Cumple la regla (< 0.005), con un modelo más simple y más explicable.
- **No es mejor, es igual.** En la validación cruzada dentro de train salió un poco peor (0.984 frente a 0.982).
  Las dos diferencias son de milésimas, muy por debajo del ruido que medimos en el notebook 06 (IC 95 % de
  ±0.009 para diferencias de este tipo). Quitar `home_elo`/`away_elo` no pierde capacidad predictiva, y a
  cambio el Elo queda en un solo coeficiente (H − A = 0.95 por desviación estándar, ≈ 2.6 veces las odds
  de *gana local* frente a *gana visitante*).
- **Sigue siendo el mejor modelo propio en validación** (por delante de Poisson y de la logística solo con
  Elo), pero queda a 0.017 de las cuotas.
- **Limitación para explicar:** los coeficientes de forma (puntos, goles a favor y en contra de los últimos 5)
  siguen correlacionados entre sí y con el Elo, y algunos signos salen contraintuitivos. En los `factores()`
  de un partido, el Elo domina cuando la diferencia de nivel es grande; cuando los equipos son parejos (como
  Arsenal vs Manchester City), aparecen arriba factores de forma con impactos pequeños (< 0.2) y signos que no
  conviene sobreinterpretar.
- `rest_diff` también es redundante (`home_rest − away_rest`), pero los coeficientes de descanso son ≈ 0, así
  que no afecta a la explicación.
